# 09 Final PRD Validation & Baseline Verification
**Project:** E-Commerce Product Analytics ? Search & Conversion Funnel  
**Workflow:** Final PRD & Product Specification Validation  
**Objective:** Programmatically audit and validate all empirical claims, canonical baseline numbers, CSV schemas, and architectural designs specified in `docs/final_prd.md` and `docs/final_product_spec.md`.


In [ ]:
import os, sys, math, json
import duckdb
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import Image, display

DB_PATH = os.path.join('..', 'data', 'ecommerce_analytics.duckdb')
conn = duckdb.connect(DB_PATH, read_only=True)

def q(sql):
    return conn.execute(sql).df()

print("Connected to DuckDB successfully!")


## 1. Canonical Baseline Verification (DuckDB Audit)
Verify total searches, specific vs. short query volumes, zero-result rates (ZRR), and search-to-PDP click-through rates (CTR).


In [ ]:
df_baseline = q('''
WITH tokens AS (
    SELECT 
        ARRAY_LENGTH(STRING_SPLIT(TRIM(query_text), ' ')) AS token_count,
        results_count,
        CASE WHEN is_zero_result THEN 1 ELSE 0 END AS is_zero_result,
        CASE WHEN has_pdp_click THEN 1 ELSE 0 END AS has_pdp_click,
        CASE WHEN reformulated_in_session THEN 1 ELSE 0 END AS reformulated_in_session
    FROM search_events
)
SELECT 
    CASE WHEN token_count >= 4 THEN 'Specific (4+ tokens)' ELSE 'Short (1-3 tokens)' END AS query_segment,
    COUNT(*) AS total_searches,
    SUM(is_zero_result) AS zero_results,
    ROUND(AVG(is_zero_result) * 100, 2) AS zrr_pct,
    SUM(has_pdp_click) AS pdp_clicks,
    ROUND(AVG(has_pdp_click) * 100, 2) AS ctr_pct,
    SUM(reformulated_in_session) AS reformulations,
    ROUND(AVG(reformulated_in_session) * 100, 2) AS reform_pct
FROM tokens
GROUP BY 1 ORDER BY 1;
''')
display(df_baseline)


## 2. Canonical Experiment Eligibility & Subgroup Analysis
Eligibility criteria: Query contains $\ge 4$ tokens AND primary strict retrieval returns $< 3$ in-stock products.
Decomposes into:
- **Subgroup A:** 0 results (898 events, 95.4%)
- **Subgroup B:** 1?2 results (43 events, 4.6%)


In [ ]:
df_eligibility = q('''
WITH tokens AS (
    SELECT 
        session_id, user_id, query_text,
        ARRAY_LENGTH(STRING_SPLIT(TRIM(query_text), ' ')) AS token_count,
        results_count,
        CASE WHEN is_zero_result THEN 1 ELSE 0 END AS is_zero_result,
        CASE WHEN has_pdp_click THEN 1 ELSE 0 END AS has_pdp_click
    FROM search_events
    WHERE ARRAY_LENGTH(STRING_SPLIT(TRIM(query_text), ' ')) >= 4 AND results_count < 3
)
SELECT 
    CASE WHEN results_count = 0 THEN 'Subgroup A: 0 Results' ELSE 'Subgroup B: 1-2 Results' END AS subgroup,
    COUNT(*) AS search_count,
    ROUND(COUNT(*) * 100.0 / 941, 2) AS pct_of_eligible,
    SUM(has_pdp_click) AS pdp_clicks,
    ROUND(AVG(has_pdp_click) * 100, 2) AS ctr_pct
FROM tokens
GROUP BY 1 ORDER BY 1;
''')
display(df_eligibility)


## 3. Session Reach & Macro Conversion Audit
Quantify the population exposed to 4+ token searches across the entire marketplace.


In [ ]:
df_reach = q('''
WITH s_flags AS (
    SELECT 
        s.session_id,
        MAX(CASE WHEN ARRAY_LENGTH(STRING_SPLIT(TRIM(se.query_text), ' ')) >= 4 THEN 1 ELSE 0 END) AS has_4plus,
        MAX(CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END) AS has_order
    FROM sessions s
    JOIN search_events se ON s.session_id = se.session_id
    LEFT JOIN orders o ON s.session_id = o.session_id
    GROUP BY 1
)
SELECT 
    CASE WHEN has_4plus = 1 THEN '4+ Token Search Sessions' ELSE 'Other Search Sessions' END AS segment,
    COUNT(*) AS sessions,
    SUM(has_order) AS orders,
    ROUND(AVG(has_order) * 100, 2) AS order_cr_pct
FROM s_flags
GROUP BY 1;
''')
display(df_reach)


## 4. Final PRD Artifacts: Requirements & Metrics Dictionary
Inspect the generated requirements catalog and metrics dictionary.


In [ ]:
df_req = pd.read_csv('../reports/final_requirements.csv')
display(df_req[['ID', 'Requirement', 'Type', 'Priority']].head(10))

df_met = pd.read_csv('../reports/final_metrics_dictionary.csv')
display(df_met[['Metric_Name', 'Category', 'Baseline_Value', 'Target_Direction']])


## 5. Statistical Power Analysis & Sample Sizing (EXP-01)
Calculate required sample size per variant for a two-tailed proportion test on Search-to-PDP CTR:
- Significance level $lpha = 0.05$ ($Z_{lpha/2} = 1.96$)
- Power $1 - eta = 0.80$ ($Z_eta = 0.84$)
- Baseline CTR $p_1 = 0.6295$


In [ ]:
def calc_sample_size_two_prop(p1, mde, alpha=0.05, power=0.80):
    p2 = p1 + mde
    p_bar = (p1 + p2) / 2.0
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    num = (z_alpha * math.sqrt(2 * p_bar * (1 - p_bar)) + z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    den = (p2 - p1) ** 2
    return math.ceil(num / den)

# Corrected experimental baseline for eligible cohort (<3 hits)
p1_eligible = 0.0308
daily_eligible = 941 / 60.0  # ~15.68 searches/day
mdes = [0.01, 0.02, 0.03, 0.035, 0.04, 0.05]

rows = []
for m in mdes:
    n_arm = calc_sample_size_two_prop(p1_eligible, m)
    total_n = n_arm * 2
    days = total_n / daily_eligible
    rows.append({
        'MDE (abs)': f'+{m*100:.1f} pp',
        'Target CTR (p2)': f'{(p1_eligible + m)*100:.2f}%',
        'Relative Lift': f'+{(m / p1_eligible)*100:.1f}%',
        'Sample / Arm': f'{n_arm:,}',
        'Total Sample': f'{total_n:,}',
        'Est. Days (@15.7/day)': round(days, 1),
        'Est. Weeks': round(days / 7.0, 1)
    })
df_power_corrected = pd.DataFrame(rows)
display(df_power_corrected)


## 6. Visual Product Artifacts (Figures 19?22)
Review the four production architecture and experiment diagrams.


In [ ]:
print("Figure 19: Proposed MVP Product Architecture")
display(Image('../reports/figures/19_final_product_architecture.png', width=900))

print("Figure 20: Final Success Metric Tree")
display(Image('../reports/figures/20_final_metric_tree.png', width=900))

print("Figure 21: MVP User Flow (Before vs After)")
display(Image('../reports/figures/21_mvp_user_flow.png', width=900))

print("Figure 22: Controlled A/B Experiment Architecture")
display(Image('../reports/figures/22_experiment_design.png', width=900))


## 7. Automated PRD Validation Suite Execution
Run `src/final_prd_validation.py` to confirm all 23 consistency and data integrity checks pass.


In [ ]:
import subprocess
result = subprocess.run([sys.executable, '../src/final_prd_validation.py'], capture_output=True, text=True)
print(result.stdout)
conn.close()
